# Qwen2.5-3B Base · SFT на Spider Expanded

1. Импортируйте этот .ipynb в Kaggle. В настройках ноутбука установите **Accelerator → GPU T4** (подойдёт T4 x2, ноутбук использует GPU 0) и включите **Internet**. Сохраните следующую версию с этими настройками.
2. Через **Add Input** подключите приватный Kaggle Dataset или Output ноутбука восстановления с файлом **spider_expanded_restored.zip**. Поддерживается и распакованная папка с manifest.json. Если найдено несколько копий, задайте точный путь в DATA_INPUT. При наличии SHA‑256 ZIP из вывода восстановления укажите его в EXPECTED_DATA_ZIP_SHA256.
3. Выполните **Run All** с MODE = "smoke". Два шага оптимизатора на самых длинных train-примерах и четыре val-примера проверят маску loss, FP16, память, градиенты и сохранение адаптера. Откройте итоговую ячейку с loss и пиком VRAM.
4. После успешного smoke смените MODE на "full". Для наблюдения за графиками перезапустите интерактивную сессию и нажмите **Run All**. Для фонового запуска выберите **Save Version → Save & Run All**. Оба варианта повторяют smoke и начинают полное обучение с чистого адаптера. После завершения скачайте ZIP и его .sha256 из **Output**.

Используются 8 222 train, 434 raw val и те же 300 model val, которые получаются через shuffle(seed=42).select(range(300)). Проверяются файлы восстановления, исторические размеры и отсутствие пересечения баз. Для побайтовой сверки с исходным историческим прогоном нужны оригинальные SHA‑256 JSONL.

По сравнению с предыдущими прогонами сохраняются SQL-промпты, 2 эпохи, seed 42, LR 1e-4, r16/alpha32/dropout 0.05 и эффективный batch 16. У 3B замороженная база загружается через QLoRA NF4 с двойным квантованием. На T4 вычисления выполняются в FP16. Microbatch 1 × накопление 16, val batch 1, gradient checkpointing. Предел контекста 1024. Для длинных примеров обрезается начало промпта, с сохранением вопроса, полного SQL и EOS. Журнал фиксирует число таких примеров.

In [ ]:
MODE = "smoke"  # "smoke" или "full"
DATA_INPUT = None  # Или "/kaggle/input/.../spider_expanded_restored.zip".
EXPECTED_DATA_ZIP_SHA256 = None  # Из вывода ноутбука восстановления, если сохранён.
MAX_LENGTH = 1024  # При OOM попробуйте 768 и повторите smoke в новой сессии.
SHOW_LIVE_PLOTS = True  # Графики обновляются в интерактивной сессии Kaggle.
RUN_SUFFIX = ""  # Например, "len768". Сохраните тот же суффикс в full.

## Среда и фиксированные версии

Код проекта загружается по тому же коммиту, что в ноутбуке 1.5B. Версии библиотек SFT совпадают с его успешным прогоном. Версии Accelerate и bitsandbytes зафиксированы отдельно. PyTorch и драйвер берутся из образа Kaggle и записываются в manifest.json. [Код проекта](https://github.com/Pavel-V418/BaseModel-vs-SFT/commit/9b02b741ae880794c09f48dc76ddd609aa5b82cc) · [QLoRA и NF4](https://huggingface.co/docs/peft/developer_guides/quantization)

In [ ]:
import gc
import hashlib
import json
import logging
import math
import os
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from contextlib import contextmanager
from importlib.metadata import version
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZIP_DEFLATED, ZipFile

if MODE not in {"smoke", "full"}:
    raise ValueError('Укажите MODE = "smoke" или "full".')
if MAX_LENGTH not in {768, 1024}:
    raise ValueError("Допустимый MAX_LENGTH — 768 или 1024.")
if RUN_SUFFIX and not re.fullmatch(r"[A-Za-z0-9_-]+", RUN_SUFFIX):
    raise ValueError("В RUN_SUFFIX допустимы буквы A–Z, цифры, _ и -.")

if "torch" in sys.modules:
    raise RuntimeError("Для настройки CUDA начните с чистой сессии Kaggle и запустите Run All.")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

ROOT = Path("/kaggle/working")
INPUT_ROOT = Path("/kaggle/input")
CACHE = Path("/kaggle/temp") / "spider_3b_sft"
CACHE.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(CACHE / "hf")
os.environ["HF_HUB_DISABLE_TELEMETRY"] = "1"

REPO_URL = "https://github.com/Pavel-V418/BaseModel-vs-SFT.git"
CODE_COMMIT = "9b02b741ae880794c09f48dc76ddd609aa5b82cc"
EVAL_COMMIT = "1bf6018f71a58551ca6f66077c7a1839a68f1f88"
MODEL_ID = "Qwen/Qwen2.5-3B"
MODEL_REVISION = "3aab1f1954e9cc14eb9509a215f9e5ca08227a9b"
SPIDER_ARCHIVE_SHA256 = "00636695dabed6b5f4b8328a16b13e069a2f16591d5efcce57660669c85b121b"
SEED = 42
MICROBATCH = 1
ACCUMULATION = 16
SUFFIX = "_nfull_others_r16_lr1e-4_s42_2ep"
BASE_RUN = "sft_3b" + SUFFIX + (f"_{RUN_SUFFIX}" if RUN_SUFFIX else "")
RESULT_ROOT = ROOT / (f"smoke_{BASE_RUN}" if MODE == "smoke" else BASE_RUN)
REPO = CACHE / f"repo-{CODE_COMMIT[:8]}"

def command(*args, **kwargs):
    return subprocess.run([str(arg) for arg in args], check=True, text=True, **kwargs)

def check_kaggle_cuda():
    # Отдельный процесс проверяет PyTorch до установки зависимостей в этой сессии.
    probe = subprocess.run(
        [sys.executable, "-c",
         "import sys, torch\n"
         "if not torch.cuda.is_available(): sys.exit(2)\n"
         "print(torch.cuda.get_device_name(0), flush=True)"],
        text=True, capture_output=True,
    )
    if probe.returncode == 2:
        detail = probe.stderr.strip()
        raise RuntimeError(
            "CUDA недоступна в этом запуске Kaggle. В настройках выберите "
            "Accelerator → GPU T4 (или GPU T4 x2), перезапустите сессию и запустите "
            "ноутбук заново. При Save & Run All сохраните версию с включённой GPU."
            + (f"\nPyTorch: {detail}" if detail else "")
        )
    if probe.returncode:
        raise RuntimeError(f"Ошибка проверки PyTorch/CUDA:\n{probe.stderr.strip()}")
    print("GPU для запуска:", probe.stdout.strip())

check_kaggle_cuda()
if not (REPO / ".git").is_dir():
    REPO.mkdir(exist_ok=True)
    command("git", "init", "-q", REPO)
head = subprocess.run(["git", "rev-parse", "--verify", "HEAD"], cwd=REPO,
                      text=True, capture_output=True)
if head.stdout.strip() != CODE_COMMIT:
    command("git", "fetch", "--depth=1", REPO_URL, CODE_COMMIT, cwd=REPO)
    command("git", "checkout", "--detach", "FETCH_HEAD", cwd=REPO)
if command("git", "rev-parse", "HEAD", cwd=REPO, capture_output=True).stdout.strip() != CODE_COMMIT:
    raise RuntimeError("Коммит кода проекта не совпадает.")
if command("git", "status", "--porcelain", "--untracked-files=no", cwd=REPO,
           capture_output=True).stdout.strip():
    raise RuntimeError("Файлы кода проекта изменены. Начните с чистой сессии.")

PINS = {
    "transformers": "5.20.2", "trl": "1.18.1", "peft": "0.21.3",
    "datasets": "4.8.5", "accelerate": "1.15.0", "bitsandbytes": "0.49.2",
    "PyYAML": "6.0.3", "matplotlib": "3.10.8",
}
constraint = CACHE / "torch-constraint.txt"
constraint.write_text(f"torch=={version('torch')}\n", encoding="utf-8")
command(sys.executable, "-m", "pip", "install", "-q", "-c", constraint,
        *(f"{package}=={number}" for package, number in PINS.items()))
# В предыдущем ноутбуке отключали необязательный torchao для совместимости CUDA.
command(sys.executable, "-m", "pip", "uninstall", "-q", "-y", "torchao")
sys.path.insert(0, str(REPO))

import torch
from datasets import Dataset, load_dataset
from peft import get_peft_model, prepare_model_for_kbit_training
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from trl import SFTTrainer
from src.plots import plot_history
from src.train import LiveLogCallback, build_peft_config, build_sft_config, load_data
from src.utils import load_config, set_seed

if not torch.cuda.is_available() or torch.cuda.device_count() != 1:
    raise RuntimeError("Включите GPU в Kaggle. Python должен видеть одну CUDA GPU.")
props = torch.cuda.get_device_properties(0)
if (props.major, props.minor) < (7, 5):
    raise RuntimeError(f"Выберите T4 или более новую GPU. Получено {props.name}.")
def read_driver_version():
    nvidia_smi = shutil.which("nvidia-smi")
    if not nvidia_smi:
        return None
    try:
        info = subprocess.run(
            [nvidia_smi, "-i", "0", "--query-gpu=driver_version", "--format=csv,noheader"],
            text=True, capture_output=True, check=True, timeout=5,
        )
    except (OSError, subprocess.SubprocessError):
        return None
    return info.stdout.strip() or None

ENVIRONMENT = {
    "python": sys.version.split()[0], "torch": torch.__version__, "cuda": torch.version.cuda,
    "gpu": props.name, "compute_capability": [props.major, props.minor],
    "gpu_total_bytes": props.total_memory, "driver_version": read_driver_version(),
    "packages": {package: version(package) for package in PINS},
}
print(json.dumps(ENVIRONMENT, indent=2))

## Восстановленный Spider Expanded

Архив восстановления содержит raw val из 434 строк, выбранные 300 строк и Spider dev из 1034 строк. Dev проверяется на пересечение баз и остаётся вне обучения. Spider 1.0 test из 2147 вопросов используется отдельно в evaluator v3.

In [ ]:
DATA_FILES = ("train.jsonl", "val.jsonl", "val_used_300.jsonl", "test.jsonl", "stats.json")

def sha256(path):
    with path.open("rb") as handle:
        return hashlib.file_digest(handle, "sha256").hexdigest()

def write_json(path, obj):
    path.write_text(json.dumps(obj, ensure_ascii=False, indent=2, allow_nan=False) + "\n",
                    encoding="utf-8")

def find_data_input():
    if DATA_INPUT:
        return Path(DATA_INPUT)
    archives = sorted(INPUT_ROOT.rglob("spider_expanded_restored.zip"))
    candidates = archives or [
        path.parent for path in sorted(INPUT_ROOT.rglob("manifest.json"))
        if all((path.parent / name).is_file() for name in DATA_FILES)
    ]
    if len(candidates) != 1:
        raise RuntimeError(f"Нужен один архив восстановления или DATA_INPUT. Найдено: {candidates}")
    return candidates[0]

@contextmanager
def open_data_input(source):
    if source.is_dir():
        yield lambda name: (source / name).open("rb")
    else:
        with ZipFile(source) as archive:
            for name in (*DATA_FILES, "manifest.json"):
                if archive.namelist().count(name) != 1:
                    raise ValueError(f"В ZIP ожидается один файл {name} в корне.")
            yield archive.open

def stage_dataset(source, target):
    if EXPECTED_DATA_ZIP_SHA256 and source.is_dir():
        raise ValueError("Для проверки SHA‑256 ZIP укажите путь к архиву в DATA_INPUT.")
    archive_sha = None if source.is_dir() else sha256(source)
    if EXPECTED_DATA_ZIP_SHA256 and archive_sha != EXPECTED_DATA_ZIP_SHA256.lower():
        raise ValueError(f"SHA‑256 входного ZIP не совпадает: {archive_sha}")
    target.mkdir(parents=True, exist_ok=True)
    checks = {}
    with open_data_input(source) as open_input:
        with open_input("manifest.json") as handle:
            original_manifest = handle.read()
        manifest = json.loads(original_manifest)
        expected_parameters = {"val_size_raw": 400, "seed": SEED, "include_others": True,
                               "model_val_size": 300}
        if (manifest.get("source_repository") != "Pavel-V418/BaseModel-vs-SFT"
                or manifest.get("source_git_commit") != CODE_COMMIT
                or manifest.get("source_archive_sha256") != SPIDER_ARCHIVE_SHA256
                or manifest.get("parameters") != expected_parameters
                or manifest.get("raw_counts") != {"train": 8222, "val": 434, "test": 1034}):
            raise ValueError("Манифест не соответствует восстановлению Spider Expanded для 1.5B.")
        for name in DATA_FILES:
            dest = target / name
            with open_input(name) as handle, dest.open("wb") as output:
                shutil.copyfileobj(handle, output)
            checks[name] = {"sha256": sha256(dest), "bytes": dest.stat().st_size}
            if checks[name] != manifest["files"][name]:
                raise ValueError(f"Размер или SHA‑256 {name} отличается от манифеста.")
        (target / "manifest.json").write_bytes(original_manifest)
    return {"input_archive_sha256": archive_sha,
            "input_manifest_sha256": hashlib.sha256(original_manifest).hexdigest(),
            "source_spider_archive_sha256": SPIDER_ARCHIVE_SHA256, "files": checks}

def read_jsonl(path):
    with path.open(encoding="utf-8") as handle:
        return [json.loads(line) for line in handle if line.strip()]

def check_dataset(folder):
    names = {"train": "train.jsonl", "raw_val": "val.jsonl", "model_val": "val_used_300.jsonl",
             "dev": "test.jsonl"}
    expected = {"train": (8222, 137), "raw_val": (434, 9),
                "model_val": (300, None), "dev": (1034, 20)}
    rows = {split: read_jsonl(folder / filename) for split, filename in names.items()}
    dbs = {split: {row["db_id"] for row in records} for split, records in rows.items()}
    for split, (count, db_count) in expected.items():
        records = rows[split]
        ids = {row["id"] for row in records}
        if len(records) != count or len(ids) != count or (db_count is not None and len(dbs[split]) != db_count):
            raise ValueError(f"Неверный размер, число уникальных id или баз для {split}.")
        if split != "dev" and any(
            not isinstance(row["prompt"], str) or not row["prompt"].endswith("SQL:")
            or not isinstance(row["completion"], str) or not row["completion"].startswith(" ")
            or not row["completion"].strip()
            for row in records
        ):
            raise ValueError(f"Неверный формат prompt/completion в {split}.")
    for first, second in (("train", "raw_val"), ("train", "dev"), ("raw_val", "dev")):
        if dbs[first] & dbs[second]:
            raise ValueError(f"Пересечение db_id между {first} и {second}.")
    stats = json.loads((folder / "stats.json").read_text(encoding="utf-8"))
    historical_dropped_dbs = {"assets_maintenance": 1, "document_management": 2}
    if stats["n_dropped_total"] != 3 or Counter(row["db_id"] for row in stats["dropped"]) != historical_dropped_dbs:
        raise ValueError("Статистика отброшенных SQL отличается от исторического прогона.")
    historical_difficulty = {
        "train": {"medium": 2831, "easy": 1897, "hard": 1805, "extra": 1689},
        "val": {"medium": 168, "hard": 116, "easy": 86, "extra": 64},
        "test": {"medium": 446, "hard": 174, "easy": 248, "extra": 166},
    }
    for split, key in (("train", "train"), ("raw_val", "val"), ("dev", "test")):
        expected_difficulty = historical_difficulty[key]
        if (Counter(row["difficulty"] for row in rows[split]) != expected_difficulty
                or stats["splits"][key]["difficulty"] != expected_difficulty):
            raise ValueError(f"Распределение сложности {split} отличается от прогона 1.5B.")

    raw_val = load_dataset("json", data_files={"validation": str(folder / "val.jsonl")})["validation"]
    selected = raw_val.shuffle(seed=SEED).select(range(300))
    if selected.to_list() != rows["model_val"]:
        raise ValueError("val_used_300.jsonl не совпадает с raw val shuffle(seed=42).select(range(300)).")
    return {split: {"rows": len(records), "dbs": len(dbs[split])} for split, records in rows.items()}

DATA_DIR = CACHE / "recovered"
DATA_SPEC = stage_dataset(find_data_input(), DATA_DIR)
DATA_SPEC["splits"] = check_dataset(DATA_DIR)

# Читаем параметры 3B из репозитория и подставляем проверенный validation subset.
CFG = load_config(REPO / "configs/sft_3b_qlora.yaml")
CFG.update({
    "run_name": BASE_RUN, "model": MODEL_ID, "model_revision": MODEL_REVISION, "seed": SEED,
    "data": {"train": str(DATA_DIR / "train.jsonl"), "val": str(DATA_DIR / "val_used_300.jsonl")},
    "train_size": None, "val_size": None,  # 300 уже выбраны один раз.
    "completion_only_loss": True, "report_to": "none", "hub_repo": None,
})
CFG["training"].update({
    "per_device_train_batch_size": MICROBATCH, "per_device_eval_batch_size": 1,
    "gradient_accumulation_steps": ACCUMULATION, "eval_steps": 50, "max_length": MAX_LENGTH,
})
raw_train, raw_val = load_data(CFG)
if (len(raw_train), len(raw_val)) != (8222, 300):
    raise RuntimeError("Тренер получил неожиданные размеры train / model val.")
print(json.dumps(DATA_SPEC, indent=2))

## Токенизация и completion-only loss

Токенизатор закреплён на ревизии модели. Используется текстовый промпт проекта с окончанием SQL:. Маска даёт loss только по SQL и EOS, включая случаи, когда ID паддинга совпадает с EOS. Примеры обрабатываются отдельно без packing.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, trust_remote_code=False)
if tokenizer.eos_token_id is None:
    raise RuntimeError("В токенизаторе отсутствует EOS.")
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

def tokenize_split(dataset, tok, max_length):
    counts = Counter()
    features = []
    for batch in dataset.iter(batch_size=256):
        prompts = batch["prompt"]
        completions = batch["completion"]
        prompt_batch = tok(prompts, add_special_tokens=False)["input_ids"]
        complete_batch = tok([a + b for a, b in zip(prompts, completions)],
                             add_special_tokens=False)["input_ids"]
        for prompt, sql, prefix, joined in zip(prompts, completions, prompt_batch, complete_batch):
            if joined[:len(prefix)] == prefix:
                target = joined[len(prefix):]
            else:
                # При слиянии токенов на границе сохраняем точный inference prompt.
                counts["separate_boundary"] += 1
                target = tok(sql, add_special_tokens=False)["input_ids"]
            if not prefix or not target or tok.eos_token_id in target:
                raise ValueError("Ожидаются непустые prompt и SQL без вложенного EOS.")
            target = target + [tok.eos_token_id]
            budget = max_length - len(target)
            if budget < 1:
                raise ValueError("Полный SQL и EOS не помещаются в max_length.")
            counts["max_original_tokens"] = max(counts["max_original_tokens"], len(prefix) + len(target))
            if len(prefix) > budget:
                question_marker = "\nВопрос: "
                if question_marker not in prompt:
                    raise ValueError("Не найден вопрос в длинном промпте.")
                question = question_marker + prompt.rsplit(question_marker, 1)[-1]
                suffix_ids = tok(question, add_special_tokens=False)["input_ids"]
                if len(suffix_ids) + 2 > budget:
                    raise ValueError("Вопрос и полный SQL не помещаются в max_length.")
                prefix = prefix[-budget:]
                if question not in tok.decode(prefix, clean_up_tokenization_spaces=False):
                    raise ValueError("Обрезка промпта затронула текст вопроса.")
                counts["left_truncated"] += 1
            features.append({"input_ids": prefix + target, "labels": [-100] * len(prefix) + target})
    counts["max_used_tokens"] = max(len(row["input_ids"]) for row in features)
    counts["left_truncated"] += 0
    counts["separate_boundary"] += 0
    return Dataset.from_list(features), dict(counts)

train_ds, train_token_stats = tokenize_split(raw_train, tokenizer, MAX_LENGTH)
val_ds, val_token_stats = tokenize_split(raw_val, tokenizer, MAX_LENGTH)
TOKEN_STATS = {"train": train_token_stats, "model_val": val_token_stats}
print(json.dumps(TOKEN_STATS, indent=2))

## Живые графики обучения

При `SHOW_LIVE_PLOTS = True` ячейка запуска показывает loss, точность токенов, learning rate и grad norm. Первый график появится после начальной оценки, дальше обновится на каждом событии логирования: каждые 10 шагов обучения и на каждой валидации в режиме full. График показывает номер шага и эпоху. Используйте **Run All** в открытой сессии Kaggle. При фоновом **Save & Run All** прогресс можно смотреть в текстовом логе, а итоговый PNG будет в ZIP.

In [ ]:
from io import BytesIO
from IPython.display import Image, display
import matplotlib.pyplot as plt

class LiveTrainingPlot:
    def __init__(self):
        self.handle = None
        self.enabled = SHOW_LIVE_PLOTS

    def refresh(self, history_path, logger):
        if not self.enabled:
            return
        try:
            figure = plot_history(history_path)
            try:
                with BytesIO() as png:
                    figure.savefig(png, format="png", dpi=90)
                    frame = Image(data=png.getvalue(), format="png")
            finally:
                plt.close(figure)
            if self.handle is None:
                self.handle = display(frame, display_id=True)
            else:
                self.handle.update(frame)
        except Exception as error:
            self.enabled = False
            logger.warning("Живой график отключён: %s. Метрики продолжают записываться.", error)

## Обучение и контроль val loss

Полный запуск оценивает 300 val-примеров перед обучением, каждые 50 шагов, в конце каждой эпохи и после загрузки лучшего адаптера. Контрольные точки выбираются по минимальному eval_loss. Если loss или grad norm становятся NaN или Inf, прогон прерывается и сохраняет лог. При CUDA OOM начните с новой сессии, установите MAX_LENGTH = 768 и повторите smoke. Используйте ту же длину в full и сохраните её при сравнении экспериментов.

In [ ]:
def select_stress_examples(dataset, count):
    sizes = [(len(labels), sum(label != -100 for label in labels))
             for labels in dataset["labels"]]
    indices = sorted(range(len(sizes)), key=sizes.__getitem__, reverse=True)[:count]
    return dataset.select(indices)

def verify_collator(trainer, train):
    # Проверяем реальный batch и отдельный batch с динамическим паддингом.
    lengths = [len(ids) for ids in train["input_ids"]]
    shortest = min(range(len(lengths)), key=lengths.__getitem__)
    longest = max(range(len(lengths)), key=lengths.__getitem__)
    features = [train[longest], train[shortest]]
    batch = trainer.data_collator(features)
    for i, row in enumerate(features):
        size = len(row["input_ids"])
        assert batch["input_ids"][i, :size].tolist() == row["input_ids"]
        assert batch["labels"][i, :size].tolist() == row["labels"]
        assert bool(batch["attention_mask"][i, :size].all())
        assert not bool(batch["attention_mask"][i, size:].any())
        assert bool((batch["labels"][i, size:] == -100).all())
        first_target = next(j for j, value in enumerate(row["labels"]) if value != -100)
        assert 0 < first_target < size - 1
        assert row["labels"][first_target:] == row["input_ids"][first_target:]
        assert row["labels"][-1] == tokenizer.eos_token_id

class RunLog(LiveLogCallback):
    def __init__(self, path, run_name, logger, steps_per_epoch):
        super().__init__(path, run_name)
        self.logger = logger
        self.steps_per_epoch = steps_per_epoch
        self.chart = LiveTrainingPlot()

    def on_log(self, args, state, control, logs=None, **kwargs):
        super().on_log(args, state, control, logs=logs, **kwargs)
        if state.is_world_process_zero and logs:
            self.logger.info("step=%d | %s | %s", state.global_step, self.phase, logs)
            for name in ("loss", "eval_loss", "grad_norm", "train_loss"):
                if name in logs and not math.isfinite(logs[name]):
                    raise FloatingPointError(f"{name} не конечен на шаге {state.global_step}.")
            if "loss" in logs or "eval_loss" in logs:
                self.chart.refresh(self.path, self.logger)

    def on_step_end(self, args, state, control, **kwargs):
        if state.global_step == state.max_steps or (
            self.steps_per_epoch and state.global_step % self.steps_per_epoch == 0
        ):
            control.should_evaluate = True
            control.should_save = True

def make_model(cfg, args):
    precision = torch.bfloat16 if args.bf16 else torch.float16
    quant = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=precision,
    )
    base = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, revision=MODEL_REVISION, dtype=precision, quantization_config=quant,
        device_map={"": 0}, attn_implementation="sdpa", trust_remote_code=False,
    )
    if not base.is_loaded_in_4bit or any(param.device.type != "cuda" for param in base.parameters()):
        raise RuntimeError("База должна быть полностью загружена в 4-bit на GPU 0.")
    base.config.use_cache = False
    base = prepare_model_for_kbit_training(
        base, use_gradient_checkpointing=True,
        gradient_checkpointing_kwargs={"use_reentrant": False},
    )
    return get_peft_model(base, build_peft_config(cfg), revision=MODEL_REVISION)

def align_adapter_precision(model, *, fp16):
    trainable = [(name, param) for name, param in model.named_parameters()
                 if param.requires_grad]
    if not trainable or any("lora_" not in name for name, _ in trainable):
        raise RuntimeError("Обучаемыми должны быть только параметры LoRA.")
    if fp16:
        # TRL 1.18.1 переводит параметры QLoRA в BF16 при создании SFTTrainer.
        # На T4 FP16 GradScaler должен получить FP32 градиенты этих параметров.
        with torch.no_grad():
            for _, param in trainable:
                if param.dtype != torch.float32:
                    param.data = param.detach().to(dtype=torch.float32)
    return sorted({str(param.dtype) for _, param in trainable})

def fit(phase, root, logger):
    smoke = phase == "smoke"
    run_name = f"smoke_{BASE_RUN}" if smoke else BASE_RUN
    train = select_stress_examples(train_ds, 2 * ACCUMULATION) if smoke else train_ds
    val = select_stress_examples(val_ds, 4) if smoke else val_ds
    steps_per_epoch = math.ceil(len(train) / (MICROBATCH * ACCUMULATION))
    cfg = {**CFG, "run_name": run_name, "training": dict(CFG["training"])}
    if smoke:
        cfg["training"].update(max_steps=2, eval_steps=1, warmup_steps=0, logging_steps=1)

    results = root / "results"
    final_dir = root / "outputs" / run_name / "final"
    scratch = CACHE / "checkpoints" / run_name
    scratch.mkdir(parents=True, exist_ok=True)
    checkpoint_dir = Path(mkdtemp(prefix=f"{phase}-", dir=scratch))
    args = build_sft_config(cfg, checkpoint_dir)
    args.dataset_kwargs = {"skip_prepare_dataset": True}
    args.prediction_loss_only = True
    args.logging_nan_inf_filter = False
    args.save_only_model = True
    args.disable_tqdm = True

    logger.info("Код %s | модель %s@%s", CODE_COMMIT, MODEL_ID, MODEL_REVISION)
    logger.info("%s | train=%s val=%s | batch=%s × %s | max_length=%s",
                phase, len(train), len(val), MICROBATCH, ACCUMULATION, MAX_LENGTH)
    set_seed(SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    model = make_model(cfg, args)

    live_path = results / f"{run_name}_live.json"
    live = RunLog(live_path, run_name, logger, None if smoke else steps_per_epoch)
    trainer = SFTTrainer(
        model=model, args=args, train_dataset=train, eval_dataset=val,
        processing_class=tokenizer, callbacks=[live],
    )
    adapter_dtypes = align_adapter_precision(trainer.model, fp16=args.fp16)
    trainable, total = trainer.model.get_nb_trainable_parameters()
    logger.info("LoRA params=%s / %s | dtype=%s | fp16=%s bf16=%s",
                trainable, total, ", ".join(adapter_dtypes), args.fp16, args.bf16)
    verify_collator(trainer, train_ds if smoke else train)
    live.dump()
    before = trainer.evaluate()
    started = time.monotonic()
    trained = trainer.train()
    elapsed = time.monotonic() - started

    best_path = trainer.state.best_model_checkpoint
    best_metric = trainer.state.best_metric
    actual_steps = trainer.state.global_step
    expected_steps = 2 if smoke else 2 * steps_per_epoch
    if actual_steps != expected_steps:
        raise RuntimeError(f"Ожидалось {expected_steps} шагов, получено {actual_steps}.")
    eval_history = [row for row in live.history if row["phase"] == "train" and "eval_loss" in row]
    if (not best_path or best_metric is None or not eval_history
            or eval_history[-1]["step"] != actual_steps
            or best_metric != min(row["eval_loss"] for row in eval_history)):
        raise RuntimeError("Не удалось подтвердить выбор лучшего checkpoint по eval_loss.")

    # Trainer уже загрузил лучший checkpoint. Повторяем оценку на той же val.
    live.phase = "after"
    after = trainer.evaluate()
    if not math.isclose(best_metric, after["eval_loss"], rel_tol=1e-4, abs_tol=1e-5):
        raise RuntimeError("Loss после загрузки лучшего checkpoint отличается от ожидаемого.")
    trainer.save_model(str(final_dir))
    tokenizer.save_pretrained(final_dir)
    adapter_files = ("adapter_config.json", "adapter_model.safetensors")
    adapter_hashes = {name: sha256(final_dir / name) for name in adapter_files}
    from safetensors.torch import load_file
    final_weights = load_file(final_dir / adapter_files[1])
    best_weights = load_file(Path(best_path) / adapter_files[1])
    if final_weights.keys() != best_weights.keys() or any(
        not torch.equal(tensor, best_weights[name]) for name, tensor in final_weights.items()
    ):
        raise RuntimeError("Веса финального адаптера не совпадают с лучшим checkpoint.")
    adapter_cfg = json.loads((final_dir / "adapter_config.json").read_text(encoding="utf-8"))
    if (adapter_cfg.get("base_model_name_or_path"), adapter_cfg.get("revision")) != (MODEL_ID, MODEL_REVISION):
        raise RuntimeError("Идентификатор базы или ревизия в adapter_config.json отличаются.")

    torch.cuda.synchronize()
    memory = {
        "peak_allocated_bytes": torch.cuda.max_memory_allocated(),
        "peak_reserved_bytes": torch.cuda.max_memory_reserved(),
        "total_bytes": torch.cuda.get_device_properties(0).total_memory,
    }
    summary = {
        "run_name": run_name, "phase": phase, "config": cfg, "env": ENVIRONMENT,
        "train_size": len(train), "val_size": len(val), "raw_val_size": 434,
        "params": {"trainable": trainable, "total": total, "adapter_dtypes": adapter_dtypes},
        "code_commit": CODE_COMMIT, "model_revision": MODEL_REVISION,
        "eval_before": before, "eval_after": after, "train_metrics": trained.metrics,
        "best_checkpoint": Path(best_path).name, "best_eval_loss": best_metric,
        "steps": actual_steps, "train_time_sec": round(elapsed, 1),
        "adapter_dir": final_dir.relative_to(root).as_posix(), "adapter_sha256": adapter_hashes,
        "memory": memory,
    }
    write_json(results / f"{run_name}_train.json", summary)
    write_json(results / f"{run_name}_log_history.json", trainer.state.log_history)
    live.phase = "done"
    live.dump()
    live.chart.refresh(live.path, logger)
    plt.close(plot_history(live_path, out=results / f"{run_name}_curves.png"))
    logger.info("Лучший %s | val %.6f → %.6f | VRAM reserved %.2f GiB",
                summary["best_checkpoint"], before["eval_loss"], after["eval_loss"],
                memory["peak_reserved_bytes"] / 2**30)
    return summary

def run_training(phase, root):
    run_name = f"smoke_{BASE_RUN}" if phase == "smoke" else BASE_RUN
    results = root / "results"
    results.mkdir(parents=True, exist_ok=True)
    log_path = results / f"{run_name}_stdout.log"
    logger = logging.Logger(f"sft3b.{phase}", level=logging.INFO)
    handlers = [logging.FileHandler(log_path, mode="x", encoding="utf-8"),
                logging.StreamHandler(sys.stdout)]
    for handler in handlers:
        handler.setFormatter(logging.Formatter("%(asctime)s %(levelname)s %(message)s"))
        logger.addHandler(handler)

    try:
        return fit(phase, root, logger)
    except Exception:
        logger.exception("Прогон остановлен. Лог: %s", log_path)
        raise
    finally:
        for handler in handlers:
            logger.removeHandler(handler)
            handler.close()
        gc.collect()
        torch.cuda.empty_cache()

## Запуск и Kaggle Output

В режиме smoke создаётся отдельный архив с префиксом smoke_. В режиме full ZIP включает лучший PEFT-адаптер в outputs/<run>/final/, токенизатор, график, stdout, историю train/val, версии, результаты smoke и SHA‑256 файлов. Рабочие checkpoints и исходные данные остаются во временной папке Kaggle. Пик PyTorch reserved приводится отдельно для smoke и full.

In [ ]:
def pack_result(root, destination):
    checksum_file = destination.with_suffix(".zip.sha256")
    if destination.exists() or checksum_file.exists():
        raise FileExistsError(destination)
    partial = destination.with_suffix(".zip.part")
    with ZipFile(partial, "x", compression=ZIP_DEFLATED) as archive:
        for file in sorted(root.rglob("*")):
            if file.is_file():
                archive.write(file, file.relative_to(root).as_posix())
    with ZipFile(partial) as archive:
        bad_member = archive.testzip()
        if bad_member is not None:
            raise IOError(f"Повреждён файл в ZIP: {bad_member}")
    digest = sha256(partial)
    partial.rename(destination)
    checksum_file.write_text(f"{digest}  {destination.name}\n", encoding="ascii")
    return checksum_file, digest

archive_path = ROOT / f"results_{RESULT_ROOT.name}.zip"
if RESULT_ROOT.exists() or archive_path.exists():
    raise FileExistsError("Этот запуск уже существует. Начните с чистой сессии или задайте RUN_SUFFIX.")
RESULT_ROOT.mkdir(parents=True)
shutil.copyfile(DATA_DIR / "stats.json", RESULT_ROOT / "recovery_stats.json")
shutil.copyfile(DATA_DIR / "manifest.json", RESULT_ROOT / "recovery_manifest.json")

smoke_root = RESULT_ROOT if MODE == "smoke" else RESULT_ROOT / "smoke"
smoke_summary = run_training("smoke", smoke_root)
summary = run_training("full", RESULT_ROOT) if MODE == "full" else smoke_summary

if MODE == "full":
    hashes = summary["adapter_sha256"]
    eval_model = {
        "evaluation_version": 3, "source_commit": EVAL_COMMIT, "test_questions": 2147,
        "mode": "zero", "chat_template": False, "archive": archive_path.name,
        "adapter_dir": summary["adapter_dir"], "run_name": BASE_RUN,
        "models": {"3b": [MODEL_ID, MODEL_REVISION, hashes["adapter_config.json"],
                          hashes["adapter_model.safetensors"]]},
    }
    write_json(RESULT_ROOT / "evaluator_v3_model.json", eval_model)

files = {file.relative_to(RESULT_ROOT).as_posix(): sha256(file)
         for file in sorted(RESULT_ROOT.rglob("*")) if file.is_file()}
manifest = {
    "run_name": summary["run_name"], "mode": MODE, "seed": SEED,
    "code_commit": CODE_COMMIT, "model": MODEL_ID, "model_revision": MODEL_REVISION,
    "data": DATA_SPEC, "tokenization": TOKEN_STATS, "environment": ENVIRONMENT,
    "effective_batch": MICROBATCH * ACCUMULATION, "max_length": MAX_LENGTH,
    "quantization": {"bits": 4, "type": "nf4", "double_quant": True,
                     "compute_dtype": "bf16" if props.major >= 8 else "fp16"},
    "selected_adapter": summary["adapter_dir"], "best_checkpoint": summary["best_checkpoint"],
    "best_eval_loss": summary["best_eval_loss"],
    "peak_cuda_memory": {"smoke": smoke_summary["memory"], MODE: summary["memory"]},
    "sha256": files,
}
write_json(RESULT_ROOT / "manifest.json", manifest)
checksum_file, archive_sha = pack_result(RESULT_ROOT, archive_path)
print(f"Output: {archive_path.name} | {checksum_file.name}")
print(f"SHA‑256: {archive_sha}")
print(f"Val loss: {summary['eval_before']['eval_loss']:.6f} → {summary['eval_after']['eval_loss']:.6f}")

from IPython.display import FileLink, Image, display
display(Image(filename=str(RESULT_ROOT / "results" / f"{summary['run_name']}_curves.png")))
if Path.cwd() == ROOT:
    display(FileLink(archive_path.name), FileLink(checksum_file.name))

## Дальше — Spider 1.0 test, 2 147 вопросов

1. Сверьте ZIP по файлу .sha256, например командой `sha256sum -c results_sft_3b_nfull_others_r16_lr1e-4_s42_2ep.zip.sha256`. Возьмите **full** ZIP и файл `evaluator_v3_model.json` из него.
2. В копии `kaggle_sft_spider_test.py` из текущего ноутбука сравнения добавьте в `MODELS` запись `"3b": tuple(spec["models"]["3b"])`, где `spec` — JSON из предыдущего шага. Можно вставить четыре строки из этого массива прямо в кортеж. Порядок полей — ID базы, ревизия, SHA‑256 конфига, SHA‑256 весов. Для сравнения сохраните записи 0.5b/1.5b и приложите их ZIP. Если задан RUN_SUFFIX, задайте для 3b имя из `spec["run_name"]` в `find_adapters`. Для стандартного имени поиск ZIP и каталога `final/` уже совпадает.
3. Оставьте evaluator v3 на коммите `1bf6018f71a58551ca6f66077c7a1839a68f1f88`. Запустите `prepare`, `smoke` на 24 примерах и затем `full` на всех 2 147. Для сопоставимого результата сохраните zero-shot, исходный текстовый шаблон без чата, greedy, 256 новых токенов и другие штатные параметры evaluator v3. Существующий runner загружает базу в FP16 на T4. В отчёте укажите отдельно QLoRA NF4 при обучении 3B и точность загрузки на оценке. [Код evaluator v3](https://github.com/Pavel-V418/BaseModel-vs-SFT/blob/1bf6018f71a58551ca6f66077c7a1839a68f1f88/src/evaluate.py)